# AI Language Coach — Qwen2.5-7B Colab API

Clean notebook for running **Qwen2.5-7B-Instruct in 4-bit on Google Colab** and exposing it through a temporary **FastAPI + ngrok** endpoint for your local VS Code RAG pipeline.

Run the cells **top to bottom**.

> Use a GPU runtime (T4 or better). Never share or commit your ngrok authtoken.

## 1. Check GPU

In [1]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. In Colab choose Runtime → Change runtime type → GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56


## 2. Install dependencies

Run this once. If Colab asks you to restart the session, restart it and continue from Cell 3.

In [2]:
!pip install -q -U \
    transformers \
    accelerate \
    sentencepiece \
    "bitsandbytes>=0.46.1" \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 3. Verify the environment

Do not continue until this cell shows that `bitsandbytes` imports successfully.

In [3]:
import sys
import torch
import transformers
import bitsandbytes as bnb

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Python: 3.13.15
Torch: 2.11.0+cu128
Transformers: 5.16.1
BitsAndBytes: 0.50.2
CUDA available: True
GPU: Tesla T4


## 4. Load Qwen2.5-7B-Instruct in 4-bit

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading Qwen in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print("\nQwen loaded successfully!")
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


Qwen loaded successfully!
Model: Qwen/Qwen2.5-7B-Instruct
GPU: Tesla T4


## 5. Local generation test

This confirms Qwen works **before** we add FastAPI or ngrok.

In [5]:
def generate_qwen(prompt, max_new_tokens=120):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI English Language Coach. "
                "Keep conversation natural, helpful, and concise."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

test_answer = generate_qwen(
    "Hello! I want to practice my English because I want to travel to London."
)

print(test_answer)

Hello! That's great! Practicing your English for a trip to London is a fantastic idea. What areas do you feel you need the most help with? Speaking, listening, reading, or writing?


## 6. Create the FastAPI app

The endpoint will be `POST /generate`.

In [6]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(
    title="AI Language Coach — Qwen API",
    version="1.0",
)

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 256

@app.get("/")
def home():
    return {
        "status": "online",
        "model": MODEL_NAME,
    }

@app.post("/generate")
def generate(request: GenerateRequest):
    try:
        if not request.prompt.strip():
            raise HTTPException(status_code=400, detail="Prompt cannot be empty.")

        response_text = generate_qwen(
            prompt=request.prompt,
            max_new_tokens=request.max_new_tokens,
        )

        return {"response": response_text}

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

print("FastAPI app created successfully.")

FastAPI app created successfully.


## 7. Add your ngrok authtoken

Get your token from your ngrok dashboard.

**Do not send the token to anyone and do not commit it to GitHub.**

In [9]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3Iybb8lOokoXleuhCXXaZdyUR0W_3e2B15YE31VrY4uFwLgWy"

if NGROK_AUTH_TOKEN == "3Iybb8lOokoXleuhCXXaZdyUR0W_3e2B15YE31VrY4uFwLgWyE":
    raise ValueError("Paste your real ngrok authtoken first.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok authentication configured.")

ngrok authentication configured.


## 8. Start FastAPI + ngrok

This version automatically selects a free port, which avoids the common `address already in use` problem.

In [10]:
import socket
import threading
import time
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

try:
    ngrok.kill()
except Exception:
    pass

sock = socket.socket()
sock.bind(("", 0))
PORT = sock.getsockname()[1]
sock.close()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info",
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)

public_tunnel = ngrok.connect(PORT)
API_URL = public_tunnel.public_url

print("\nQwen API is online!")
print("Local port:", PORT)
print("Public URL:", API_URL)
print("Generate endpoint:", f"{API_URL}/generate")
print("API docs:", f"{API_URL}/docs")

INFO:     Started server process [935]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:43117 (Press CTRL+C to quit)



Qwen API is online!
Local port: 43117
Public URL: https://numerous-porous-impending.ngrok-free.dev
Generate endpoint: https://numerous-porous-impending.ngrok-free.dev/generate
API docs: https://numerous-porous-impending.ngrok-free.dev/docs


## 9. Test the public Qwen endpoint

Expected result: **Status: 200** and a JSON response containing `"response"`.

In [11]:
import requests

response = requests.post(
    f"{API_URL}/generate",
    json={
        "prompt": "Say hello to an English learner and ask them one simple question.",
        "max_new_tokens": 100,
    },
    timeout=180,
)

print("Status:", response.status_code)

if response.ok:
    print("Response:", response.json())
else:
    print("Error:")
    print(response.text)

INFO:     136.67.3.101:0 - "POST /generate HTTP/1.1" 200 OK
Status: 200
Response: {'response': 'Hello! How are you doing today?'}


## 10. Test a RAG-style prompt

This simulates the final prompt that your local VS Code RAG pipeline will send to Qwen.

In [12]:
rag_test_prompt = '''
SYSTEM:
You are an AI English Language Coach.

RETRIEVED KNOWLEDGE:
Present Perfect: I have eaten sushi.
The present perfect can connect past events with the present.

RECENT CONVERSATION:
user: Can you explain the present perfect?
assistant: Sure. It connects past actions with the present.

USER MESSAGE:
When should I use it?

INSTRUCTION:
Respond naturally as an English language coach.
Use the retrieved knowledge only if it helps answer the learner.
'''.strip()

response = requests.post(
    f"{API_URL}/generate",
    json={
        "prompt": rag_test_prompt,
        "max_new_tokens": 180,
    },
    timeout=180,
)

print("Status:", response.status_code)

if response.ok:
    print("\nQwen response:")
    print(response.json()["response"])
else:
    print(response.text)

INFO:     136.67.3.101:0 - "POST /generate HTTP/1.1" 200 OK
Status: 200

Qwen response:
Sure! You use the present perfect when you want to talk about something that happened in the past but its effects are still relevant now. For example, "I have eaten sushi" means you ate sushi at some point, and now you can discuss your experience with it.
